# VRP Toolkit: Sensitivity Analysis Tutorial

Welcome to the Sensitivity Analysis tutorial! This tutorial builds upon the Quick Start tutorial and shows you how to analyze how different parameters affect your VRP solutions.

## What You'll Learn
- Design sensitivity analysis experiments for VRP problems
- Run multiple scenarios with different random seeds
- Collect and analyze key performance metrics
- Compare solutions with and without charging constraints
- Export results for further analysis

**Prerequisites:** Complete the Quick Start tutorial first to understand the basic workflow.

**Time estimate:** 5-8 minutes

## 1. Import Required Modules

We'll use the same modules as the Quick Start tutorial, plus additional tools for data collection and analysis:

In [ ]:
import numpy as np
import pandas as pd
import random
import matplotlib.pyplot as plt

# Import VRP Toolkit modules
from vrp_toolkit.data.map import RealDataMap
from vrp_toolkit.data.generators import DemandGenerator, OrderGenerator
from vrp_toolkit.problems.pdptw import PDPTWInstance
from vrp_toolkit.algorithms.alns.solver import greedy_insertion_initial_solution, ALNS, ALNSConfig

## 2. Design Your Sensitivity Analysis

Sensitivity analysis helps answer questions like:
- How sensitive is solution quality to random seed changes?
- How do different parameters affect results?
- What's the impact of charging station constraints?

**Experiment Design:**
- **Number of runs:** 10 (for demonstration; original used 60)
- **Parameters varied:** Random seeds only (keeping other parameters constant)
- **Metrics collected:** Travel distance, objective value, vehicles used, battery swaps
- **Comparison:** Solutions with vs. without charging constraints

**Note:** For production analysis, you might vary other parameters like battery capacity, number of vehicles, or time window lengths.

In [ ]:
# Experiment parameters
NUM_RUNS = 10  # Number of experiment runs (use smaller value for quick testing)

# Problem parameters (kept constant across runs)
AVERAGE_ORDER = 60  # Average number of orders
NUM_VEHICLES = 9     # Number of available vehicles
BATTERY = 8          # Battery capacity in miles

# ALNS algorithm parameters
MAX_NO_IMPROVE = 25
SEGMENT_LENGTH = 10
NUM_SEGMENTS = 15
R = 0.2  # Weight update rate
SIGMA = [10, 5, 1]  # Acceptance scores
START_TEMP = 100
COOLING_RATE = 0.99

# Other problem parameters
VEHICLE_CAPACITY = 6
BATTERY_CONSUME_RATE = 1
PENALTY_UNVISITED = 100
PENALTY_DELAYED = 5

# Map and demand parameters
TIME_RANGE = 120  # minutes
TIME_STEP = 8     # minutes
ROBOT_SPEED = 4   # mph

print(f"Experiment designed: {NUM_RUNS} runs with constant parameters")
print(f"Average orders: {AVERAGE_ORDER}, Vehicles: {NUM_VEHICLES}, Battery: {BATTERY} miles")

## 3. Define Helper Functions

These helper functions calculate battery capacity and create distance-time matrices. They're the same as in the Quick Start tutorial but included here for completeness:

In [ ]:
def battery_relaxation(battery, dist_matrix, robot_speed, indicator=None):
    """Calculate battery capacity with optional relaxation.
    
    Args:
        battery: Battery capacity in miles
        dist_matrix: Distance matrix between nodes
        robot_speed: Robot speed in mph
        indicator: If True, apply battery relaxation formula
    
    Returns:
        Battery capacity in minutes
    """
    if indicator:
        # Relaxation formula considering average depot-to-node distance
        battery_capacity = (battery - np.mean(dist_matrix[0][1:-1])) * 2 / robot_speed * 60
    else:
        battery_capacity = battery / robot_speed * 60
    return battery_capacity


def generate_d_matrix(instance):
    """Generate distance-time matrix considering time window differences.
    
    This matrix combines spatial distance with temporal proximity to help
    identify similar orders for removal operators.
    
    Args:
        instance: PDPTWInstance object
    
    Returns:
        Distance-time matrix (n x n)
    """
    n = instance.n
    robot_speed = instance.robot_speed
    dist_matrix = instance.distance_matrix
    start_time = np.array([instance.time_windows[i][0] for i in range(1, n + 1)])
    end_time = np.array([instance.time_windows[i][1] for i in range(1, n + 1)])
    
    d_matrix = np.zeros((n, n))
    for i in range(n):
        for j in range(n):
            # Calculate all four possible distance-time combinations
            # pickup-pickup
            d_1 = dist_matrix[i + 1][j + 1]
            t_1 = abs(start_time[i] - start_time[j]) / 60 * robot_speed
            dt_1 = d_1 + t_1 * 0.3

            # pickup-dropoff
            d_2 = dist_matrix[i + 1][j + n + 1]
            t_2 = abs(start_time[i] - end_time[j]) / 60 * robot_speed
            dt_2 = d_2 + t_2 * 0.3

            # dropoff-pickup
            d_3 = dist_matrix[i + n + 1][j + 1]
            t_3 = abs(start_time[j] - end_time[i]) / 60 * robot_speed
            dt_3 = d_3 + t_3 * 0.3

            # dropoff-dropoff
            d_4 = dist_matrix[i + n + 1][j + n + 1]
            t_4 = abs(start_time[j] - end_time[i]) / 60 * robot_speed
            dt_4 = d_4 + t_4 * 0.3

            # Use minimum distance-time combination
            d_matrix[i][j] = min(dt_1, dt_2, dt_3, dt_4)

    return d_matrix

## 4. Run the Sensitivity Analysis

Now we'll run the experiment loop. For each run:
1. Set a unique random seed
2. Load real map data (Purdue campus)
3. Generate demand data
4. Create PDPTW instance
5. Solve with ALNS (with and without charging constraints)
6. Collect performance metrics

**Note:** This loop may take several minutes depending on NUM_RUNS. We use NUM_RUNS=10 for demonstration.

In [ ]:
# Initialize data collection lists
results = {
    "order_count": [],        # Number of orders in instance
    "distance_uncharge": [],  # Total distance without charging
    "obj_uncharge": [],       # Objective value without charging
    "max_dist": [],           # Maximum route distance
    "distance_charge": [],    # Total distance with charging
    "obj_charge": [],         # Objective value with charging
    "num_veh": [],            # Number of vehicles used
    "battery_swapping": []    # Number of battery swaps needed
}

# Base seed value
base_seed = 42

print(f"Starting sensitivity analysis with {NUM_RUNS} runs...")

for run in range(NUM_RUNS):
    # Set unique random seed for this run
    seed_value = base_seed + run
    np.random.seed(seed_value)
    random.seed(seed_value)
    
    print(f"\nRun {run + 1}/{NUM_RUNS} (seed: {seed_value})")
    
    # ================== Load Real Map Data ==================
    # Note: These data files need to be in your working directory
    # You can download them from the Purdue campus dataset
    node_info_file = 'data/purdue_node_info.csv'
    tt_matrix = 'data/tt_matrix.csv'
    
    try:
        real_data_map = RealDataMap(node_info_file, tt_matrix)
    except FileNotFoundError as e:
        print(f"Warning: Data files not found. Using synthetic data for demonstration.")
        # For demonstration, we'll create a simple instance
        # In practice, you would need the actual data files
        from vrp_toolkit.data.map import RealMap
        real_data_map = RealMap(n_r=2, n_c=4)
    
    # ================== Generate Demand Data ==================
    n_r = real_data_map.N_R
    n_c = real_data_map.N_C
    n_pairs = n_r * n_c  # total number of restaurant-customer pairs
    n_time_intervals = TIME_RANGE / TIME_STEP
    lam_poisson = AVERAGE_ORDER / (n_pairs * n_time_intervals)

    random_params = {
        'sample_dist': {'function': np.random.randint, 'params': {'low': n_pairs - 0.5, 'high': n_pairs}},
        'demand_dist': {'function': np.random.poisson, 'params': {'lam': lam_poisson}}
    }

    restaurants = real_data_map.restaurants
    customers = real_data_map.customers
    
    # Create demands
    demands = DemandGenerator(
        TIME_RANGE,
        TIME_STEP,
        restaurants,
        customers,
        random_params=random_params
    )
    
    # ================== Create PDPTW Orders ==================
    time_params = {
        'time_window_length': 60,
        'service_time': 2,
        'extra_time': 0,
        'big_time': 1000
    }
    
    pdptw_order = OrderGenerator(
        real_data_map,
        demands.demand_table,
        time_params,
        robot_speed=ROBOT_SPEED
    )
    
    # ================== Create PDPTW Instance ==================
    pdptw_instance = PDPTWInstance(pdptw_order)
    
    # ================== Generate Initial Solution ==================
    dist_matrix = pdptw_instance.distance_matrix
    robot_speed = pdptw_instance.robot_speed
    if_battery_relaxation = 1
    battery_capacity = battery_relaxation(BATTERY, dist_matrix, robot_speed, if_battery_relaxation)

    initial_solution = greedy_insertion_initial_solution(
        pdptw_instance,
        NUM_VEHICLES,
        VEHICLE_CAPACITY,
        battery_capacity,
        BATTERY_CONSUME_RATE,
        PENALTY_UNVISITED,
        PENALTY_DELAYED
    )
    
    # ================== Configure and Run ALNS ==================
    d_matrix = generate_d_matrix(pdptw_instance)
    
    # Create ALNS configuration
    config = ALNSConfig(
        num_removal=int(pdptw_instance.n * 0.3),
        p=3,
        k=3,
        L_max=6,
        avg_remove_order=6,
        d_matrix=d_matrix,
        max_no_improve=MAX_NO_IMPROVE,
        segment_length=SEGMENT_LENGTH,
        num_segments=NUM_SEGMENTS,
        r=R,
        sigma=tuple(SIGMA),
        start_temp=START_TEMP,
        cooling_rate=COOLING_RATE
    )
    
    alns = ALNS(
        initial_solution=initial_solution,
        config=config,
        dist_matrix=dist_matrix,
        battery_capacity=battery_capacity
    )
    
    # Run ALNS algorithm
    best_solution, best_charging_solution = alns.run()
    
    # ================== Collect Results ==================
    results["order_count"].append(best_solution.instance.n)
    
    # Convert travel times to distances (minutes → miles)
    # total_travel_times are in minutes, divide by 60 and multiply by speed to get miles
    distance_uncharge = np.sum(best_solution.total_travel_times) / 60 * robot_speed
    results["distance_uncharge"].append(distance_uncharge)
    
    distance_charge = np.sum(best_charging_solution.total_travel_times) / 60 * robot_speed
    results["distance_charge"].append(distance_charge)
    
    max_dist = max(best_solution.total_travel_times) / 60 * robot_speed
    results["max_dist"].append(max_dist)
    
    results["obj_uncharge"].append(best_solution.objective_function())
    results["obj_charge"].append(best_charging_solution.objective_function())
    
    # Count number of vehicles used (non-empty routes)
    vehicles_used = len([route for route in best_charging_solution.routes if route != [0, 0]])
    results["num_veh"].append(vehicles_used)
    
    # Count battery swaps (charging station visits)
    # Charging station is typically the last node in the distance matrix
    charging_station_index = len(dist_matrix) - 1
    battery_swaps = 0
    for route in best_charging_solution.routes:
        if charging_station_index in route:
            battery_swaps += 1
    results["battery_swapping"].append(battery_swaps)
    
    print(f"  Orders: {results['order_count'][-1]}, "
          f"Distance: {distance_uncharge:.1f}→{distance_charge:.1f} miles, "
          f"Vehicles: {vehicles_used}, Battery swaps: {battery_swaps}")

print(f"\nSensitivity analysis completed! Collected data from {NUM_RUNS} runs.")

## 5. Analyze and Visualize Results

Now let's analyze the collected data to understand the variability and impact of different parameters:

In [ ]:
# Convert results to DataFrame for analysis
df_results = pd.DataFrame(results)

print("=== Summary Statistics ===")
print(df_results.describe())

print("\n=== Impact of Charging Constraints ===")
distance_diff = df_results["distance_charge"] - df_results["distance_uncharge"]
obj_diff = df_results["obj_charge"] - df_results["obj_uncharge"]
print(f"Average distance increase with charging: {distance_diff.mean():.2f} miles ({distance_diff.mean()/df_results['distance_uncharge'].mean()*100:.1f}%)")
print(f"Average objective value increase: {obj_diff.mean():.2f} ({obj_diff.mean()/df_results['obj_uncharge'].mean()*100:.1f}%)")

print("\n=== Battery Swap Analysis ===")
print(f"Runs requiring battery swaps: {(df_results['battery_swapping'] > 0).sum()} / {NUM_RUNS}")
print(f"Average battery swaps per run: {df_results['battery_swapping'].mean():.2f}")

print("\n=== Vehicle Utilization ===")
print(f"Average vehicles used: {df_results['num_veh'].mean():.1f} out of {NUM_VEHICLES} available")
print(f"Vehicle utilization: {df_results['num_veh'].mean()/NUM_VEHICLES*100:.1f}%")

## 6. Visualize Key Findings

Let's create some visualizations to better understand the results:

In [ ]:
# Create visualizations
fig, axes = plt.subplots(2, 2, figsize=(12, 10))

# Plot 1: Distance comparison
axes[0, 0].bar(['Without Charging', 'With Charging'], 
               [df_results['distance_uncharge'].mean(), df_results['distance_charge'].mean()],
               yerr=[df_results['distance_uncharge'].std(), df_results['distance_charge'].std()],
               capsize=5, color=['skyblue', 'lightcoral'])
axes[0, 0].set_ylabel('Total Distance (miles)')
axes[0, 0].set_title('Impact of Charging Constraints on Travel Distance')
axes[0, 0].grid(True, alpha=0.3)

# Plot 2: Objective value comparison
axes[0, 1].bar(['Without Charging', 'With Charging'], 
               [df_results['obj_uncharge'].mean(), df_results['obj_charge'].mean()],
               yerr=[df_results['obj_uncharge'].std(), df_results['obj_charge'].std()],
               capsize=5, color=['skyblue', 'lightcoral'])
axes[0, 1].set_ylabel('Objective Value')
axes[0, 1].set_title('Impact of Charging Constraints on Solution Quality')
axes[0, 1].grid(True, alpha=0.3)

# Plot 3: Vehicle utilization
vehicle_counts = df_results['num_veh'].value_counts().sort_index()
axes[1, 0].bar(vehicle_counts.index, vehicle_counts.values, color='lightgreen')
axes[1, 0].set_xlabel('Number of Vehicles Used')
axes[1, 0].set_ylabel('Frequency')
axes[1, 0].set_title('Vehicle Utilization Distribution')
axes[1, 0].axvline(x=NUM_VEHICLES, color='red', linestyle='--', label=f'Available: {NUM_VEHICLES}')
axes[1, 0].legend()
axes[1, 0].grid(True, alpha=0.3)

# Plot 4: Battery swaps
battery_counts = df_results['battery_swapping'].value_counts().sort_index()
axes[1, 1].bar(battery_counts.index, battery_counts.values, color='orange')
axes[1, 1].set_xlabel('Number of Battery Swaps')
axes[1, 1].set_ylabel('Frequency')
axes[1, 1].set_title('Battery Swap Requirements')
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 7. Export Results for Further Analysis

Finally, let's export the results to a CSV file for further analysis or reporting:

In [ ]:
# Export results to CSV
output_filename = f'sensitivity_results_{AVERAGE_ORDER}_orders_{NUM_VEHICLES}_vehicles.csv'
df_results.to_csv(output_filename, index=False)

print(f"Results exported to: {output_filename}")
print(f"\nFile contains {len(df_results)} rows with columns:")
for col in df_results.columns:
    print(f"  - {col}")

## Conclusion

Congratulations! You've successfully completed a sensitivity analysis for VRP problems. Here's what you've accomplished:

1. ✅ **Designed a sensitivity analysis experiment** with multiple runs and controlled parameters
2. ✅ **Implemented experiment loop** that varies random seeds while keeping other parameters constant
3. ✅ **Collected comprehensive metrics** including travel distance, objective values, vehicle usage, and battery swaps
4. ✅ **Analyzed results** to understand variability and impact of charging constraints
5. ✅ **Created visualizations** to communicate key findings effectively
6. ✅ **Exported results** for further analysis or reporting

### Key Insights from This Analysis

- **Variability:** Even with fixed parameters, random seed changes create variability in solutions
- **Charging Impact:** Battery constraints typically increase travel distance and objective values
- **Vehicle Utilization:** Understanding how many vehicles are actually needed helps with fleet planning
- **Battery Requirements:** Analysis shows how often battery swaps are needed for given parameters

### Next Steps for Further Analysis

1. **Parameter Sweeps:** Vary battery capacity, number of vehicles, or time windows systematically
2. **Statistical Analysis:** Perform ANOVA or regression to quantify parameter importance
3. **Multi-objective Analysis:** Consider trade-offs between distance, vehicle usage, and battery swaps
4. **Scenario Comparison:** Compare different problem scenarios (urban vs. rural, peak vs. off-peak)

### Practical Applications

- **Fleet Sizing:** Use sensitivity analysis to determine optimal fleet size
- **Battery Planning:** Understand battery requirements for different operational scenarios
- **Service Design:** Optimize time windows and service parameters based on sensitivity results
- **Robustness Testing:** Ensure solutions are robust to random variations in demand

### Remember

- Sensitivity analysis is a powerful tool for understanding problem behavior
- Always document your experiment design and assumptions
- Consider computational cost when designing experiments
- Use visualization to communicate findings effectively

You now have a complete framework for analyzing VRP solution sensitivity. Apply these techniques to your own problems to gain deeper insights!